# 03: Math, Statistics & Broadcasting (Exercises 31–45)

Master NumPy mathematical functions, datetime intervals, in-place allocations, and array comparison semantics.

---


In [ ]:
import numpy as np
print(f"NumPy version: {np.__version__}")

### Exercise 31: How to ignore all numpy warnings?
**Difficulty:** `★☆☆`  
**Tags:** `Error Handling, Warnings`

#### 💡 Intuition & Concept
`np.errstate(all='ignore')` provides a context manager to temporarily suppress floating point warnings like divide-by-zero or invalid value.

#### ⚠️ Key Takeaway & Gotchas
Silencing warnings globally via `np.seterr` can mask genuine numerical instability bugs. Always prefer scoped context managers with `with np.errstate(...)`.


In [ ]:
with np.errstate(all="ignore"):
    result = np.ones(5) / 0
    print("Handled with np.errstate:", result)

### Exercise 32: Is the following expression true? np.sqrt(-1) == np.emath.sqrt(-1)
**Difficulty:** `★☆☆`  
**Tags:** `Complex Numbers, Math`

#### 💡 Intuition & Concept
`np.sqrt` operates purely in real domain by default, producing `np.nan` for negative arguments. `np.emath.sqrt` automatically produces complex numbers ($1j$) when input is negative.

#### ⚠️ Key Takeaway & Gotchas
Because `np.nan == 1j` is `False`, the expression evaluates to `False`.


In [ ]:
val_real = np.sqrt(-1)
val_complex = np.emath.sqrt(-1)
print(f"np.sqrt(-1):       {val_real}")
print(f"np.emath.sqrt(-1): {val_complex}")
print(f"Equal?             {val_real == val_complex}")

### Exercise 33: How to get the dates of yesterday, today, and tomorrow?
**Difficulty:** `★☆☆`  
**Tags:** `Datetime, Time Dtypes`

#### 💡 Intuition & Concept
NumPy provides native ISO 8601 date and time representations via `np.datetime64` and duration offsets via `np.timedelta64`.

#### ⚠️ Key Takeaway & Gotchas
Specifying unit `'D'` creates calendar day precision. Adding or subtracting `np.timedelta64(1, 'D')` accurately steps calendar days.


In [ ]:
today     = np.datetime64('today', 'D')
yesterday = today - np.timedelta64(1, 'D')
tomorrow  = today + np.timedelta64(1, 'D')

print(f"Yesterday: {yesterday}")
print(f"Today:     {today}")
print(f"Tomorrow:  {tomorrow}")

### Exercise 34: How to get all the dates corresponding to the month of July 2016?
**Difficulty:** `★★☆`  
**Tags:** `Datetime, Arange`

#### 💡 Intuition & Concept
`np.arange` supports `datetime64` boundaries. Passing start `'2016-07'` and end `'2016-08'` generates every single day in the month of July.

#### ⚠️ Key Takeaway & Gotchas
The end date is exclusive, so stopping at `'2016-08'` cleanly includes July 31st.


In [ ]:
july_days = np.arange('2016-07', '2016-08', dtype='datetime64[D]')
print(f"Total days: {len(july_days)}")
print(f"First day: {july_days[0]}, Last day: {july_days[-1]}")
print(july_days[:5], "...")

### Exercise 35: How to compute ((A+B)*(-A/2)) in place (without copy)?
**Difficulty:** `★★☆`  
**Tags:** `Memory Optimization, In-place`

#### 💡 Intuition & Concept
Every ufunc has an `out=` parameter that allows redirecting output directly into an existing array buffer, avoiding the allocation of temporary arrays.

#### ⚠️ Key Takeaway & Gotchas
Standard arithmetic operators like `+` and `*` create intermediate heap-allocated temporary arrays, which can cause out-of-memory errors on massive data.


In [ ]:
A = np.ones(5) * 4
B = np.ones(5) * 2

# Target: ((A+B)*(-A/2)) in-place
np.add(A, B, out=B)          # B = A + B
np.divide(A, 2, out=A)       # A = A / 2
np.negative(A, out=A)     # A = -A
np.multiply(A, B, out=A)     # A = A * B
print("In-place result in A:", A)

### Exercise 36: Extract the integer part of a random array using 4 different methods
**Difficulty:** `★★☆`  
**Tags:** `Math, Truncation`

#### 💡 Intuition & Concept
NumPy provides multiple ways to truncate floating point numbers: modulo arithmetic (`Z - Z % 1`), floor division (`Z // 1`), mathematical functions (`np.floor`, `np.trunc`), and dtype casting (`Z.astype(int)`).

#### ⚠️ Key Takeaway & Gotchas
For negative floats, `np.floor` rounds towards $-\infty$ (e.g. $-1.5 \to -2$), while `np.trunc` and `astype(int)` round towards zero (e.g. $-1.5 \to -1$).


In [ ]:
rng = np.random.default_rng(seed=42)
Z = rng.uniform(0, 10, 5)
print("Original floats: ", np.round(Z, 3))
print("Method 1 (Z - Z % 1): ", Z - Z % 1)
print("Method 2 (Z // 1):    ", Z // 1)
print("Method 3 (np.trunc):  ", np.trunc(Z))
print("Method 4 (astype):    ", Z.astype(int))

### Exercise 37: Create a 5x5 matrix with row values ranging from 0 to 4
**Difficulty:** `★☆☆`  
**Tags:** `Broadcasting, Matrix Creation`

#### 💡 Intuition & Concept
Broadcasting rules automatically expand shapes. Adding a shape `(1, 5)` array to a `(5, 5)` array of zeros broadcasts each row with values 0 through 4.

#### ⚠️ Key Takeaway & Gotchas
You can also use `np.tile` or `np.broadcast_to`.


In [ ]:
Z = np.zeros((5, 5)) + np.arange(5)
print(Z)

### Exercise 38: Build an array from a generator yielding 10 integers
**Difficulty:** `★☆☆`  
**Tags:** `Generators, np.fromiter`

#### 💡 Intuition & Concept
`np.fromiter(iterable, dtype, count)` constructs an array directly from an arbitrary Python generator/iterator without loading all items into a Python list first.

#### ⚠️ Key Takeaway & Gotchas
Supplying the `count` parameter enables pre-allocating the exact memory buffer, significantly improving speed.


In [ ]:
def int_generator():
    for x in range(10):
        yield x * 2

Z = np.fromiter(int_generator(), dtype=int, count=10)
print(Z)

### Exercise 39: Create a vector of size 10 with values ranging from 0 to 1, both excluded
**Difficulty:** `★☆☆`  
**Tags:** `Linspace, Intervals`

#### 💡 Intuition & Concept
`np.linspace(start, stop, num)` generates evenly spaced points. Generating 11 or 12 points and slicing off the boundary endpoints excludes 0 and 1.

#### ⚠️ Key Takeaway & Gotchas
Using `endpoint=False` excludes the upper bound. Slicing `[1:]` on `linspace(0, 1, 11, endpoint=False)` excludes both.


In [ ]:
Z = np.linspace(0, 1, 11, endpoint=False)[1:]
print("Vector:", Z)
print("Count:", len(Z), "Min:", Z[0], "Max:", Z[-1])

### Exercise 40: Create a random vector of size 10 and sort it
**Difficulty:** `★☆☆`  
**Tags:** `Sorting, Algorithms`

#### 💡 Intuition & Concept
`arr.sort()` sorts the array in-place using quicksort/introsort ($O(N \log N)$). To get a sorted copy without mutating the original, use `np.sort(arr)`.

#### ⚠️ Key Takeaway & Gotchas
For high-dimensional arrays, `arr.sort(axis=...)` specifies along which axis sorting occurs.


In [ ]:
rng = np.random.default_rng(seed=10)
Z = rng.random(10)
print("Unsorted:", np.round(Z, 3))
Z.sort()
print("Sorted:  ", np.round(Z, 3))

### Exercise 41: How to sum a small array faster than np.sum?
**Difficulty:** `★★☆`  
**Tags:** `Performance, Reductions`

#### 💡 Intuition & Concept
`np.add.reduce(arr)` directly invokes the C ufunc reduction loop, bypassing the Python wrapper dispatch logic of `np.sum`.

#### ⚠️ Key Takeaway & Gotchas
The performance difference is most noticeable for small arrays in tight loops where Python dispatch overhead dominates computation time.


In [ ]:
import time
Z = np.arange(100)

t0 = time.perf_counter()
for _ in range(10000):
    s1 = np.sum(Z)
t1 = time.perf_counter()

t2 = time.perf_counter()
for _ in range(10000):
    s2 = np.add.reduce(Z)
t3 = time.perf_counter()

print(f"np.sum time:        {(t1 - t0)*1000:.2f} ms")
print(f"np.add.reduce time: {(t3 - t2)*1000:.2f} ms")
assert s1 == s2

### Exercise 42: Consider two random arrays A and B, check if they are equal
**Difficulty:** `★★☆`  
**Tags:** `Comparisons, Equality`

#### 💡 Intuition & Concept
`np.array_equal(A, B)` checks for exact identical values and shape. `np.allclose(A, B)` checks equality within numerical float tolerance.

#### ⚠️ Key Takeaway & Gotchas
Using `A == B` returns an element-wise boolean array, not a single scalar boolean!


In [ ]:
A = np.array([1.0, 2.0, 3.0])
B = np.array([1.0, 2.0, 3.0 + 1e-10])

print("Exact equal (np.array_equal):", np.array_equal(A, B))
print("Float close (np.allclose):   ", np.allclose(A, B))

### Exercise 43: Make an array immutable (read-only)
**Difficulty:** `★★☆`  
**Tags:** `Immutability, Flags`

#### 💡 Intuition & Concept
Array mutability is controlled by the `flags.writeable` boolean property. Setting `Z.flags.writeable = False` renders it read-only.

#### ⚠️ Key Takeaway & Gotchas
Any subsequent write attempt raises a `ValueError: assignment destination is read-only`.


In [ ]:
Z = np.zeros(5)
Z.flags.writeable = False

try:
    Z[0] = 42
except ValueError as e:
    print("Caught expected protection error:", e)

### Exercise 44: Convert a random 10x2 matrix representing cartesian coordinates to polar coordinates
**Difficulty:** `★★☆`  
**Tags:** `Trigonometry, Coordinate Systems`

#### 💡 Intuition & Concept
Given cartesian coordinates $(X, Y)$, radius is $R = \sqrt{X^2 + Y^2}$ (via `np.hypot(X, Y)`), and angle is $\theta = \text{atan2}(Y, X)$ (via `np.arctan2(Y, X)`).

#### ⚠️ Key Takeaway & Gotchas
Always prefer `np.hypot` over `np.sqrt(X**2 + Y**2)` to prevent floating-point underflow or overflow during squaring.


In [ ]:
rng = np.random.default_rng(seed=42)
Z = rng.random((10, 2))
X, Y = Z[:, 0], Z[:, 1]
R = np.hypot(X, Y)
Theta = np.arctan2(Y, X)

print("Radius R:    ", np.round(R[:4], 3))
print("Angle Theta: ", np.round(Theta[:4], 3))

### Exercise 45: Create random vector of size 10 and replace maximum value by 0
**Difficulty:** `★☆☆`  
**Tags:** `Indexing, Argmax`

#### 💡 Intuition & Concept
`Z.argmax()` returns the flat index of the maximum value. Setting `Z[Z.argmax()] = 0` replaces that peak element.

#### ⚠️ Key Takeaway & Gotchas
If there are multiple occurrences of the maximum, `argmax` only returns the first occurrence index.


In [ ]:
rng = np.random.default_rng(seed=99)
Z = rng.random(10)
print("Before:", np.round(Z, 3))
max_idx = Z.argmax()
print(f"Max value was {Z[max_idx]:.3f} at index {max_idx}")
Z[max_idx] = 0
print("After: ", np.round(Z, 3))